# TartanAir V2 640×640 + IMU + baro — dựng dataset trên Kaggle

Notebook này **không train**. Nó lấy một lát mỏng và đều của **cả 74 môi trường** TartanAir V2
(camera `lcam_front`, `Data_easy` + `Data_hard`), giữ ảnh **640×640 gốc**, giữ **toàn bộ IMU 100 Hz**
và thêm một **baro tổng hợp**.

| | dataset hiện tại | dataset mới (mặc định: 3 shard) |
|---|---|---|
| môi trường | 14 | **74** (đô thị, rừng, nội thất, hang, biển, đêm, sci-fi…) |
| ảnh | 256×256 | **640×640**, WebP lossless: pixel giống hệt PNG gốc |
| dung lượng | 8 GB | ~52 GB (3 × ~17 GB) |
| khung giữ lại | — | ~100 nghìn; mỗi môi trường ~1.400 (môi trường nhỏ hơn thế thì lấy hết) |
| IMU | acc, gyro 100 Hz | toàn bộ file IMU của TartanAir (acc, gyro, pos/vel/ori_global…) |
| baro | — | `imu/baro.npy`: áp suất sạch 100 Hz, cùng mốc thời gian với IMU |

Cell 2 in ra con số thật trước khi tải gì.

**Vì sao phải chia shard.** Ảnh `lcam_front` của TartanAir V2 là 833 GB PNG, còn một lần chạy Kaggle
chỉ lưu được 20 GB output. Vì vậy mỗi môi trường chỉ lấy một phần khung, **chia đều giữa các môi
trường** (môi trường nhỏ cho hết những gì nó có, phần dư chia cho môi trường khác). Trong một quỹ
đạo, các khung giữ lại cách đều nhau: trung bình ~1 s, dày hơn ở môi trường nhỏ và thưa hơn (vài
giây) ở môi trường lớn. Khung 10 Hz liền nhau gần như là bản sao, còn cách nhau cỡ giây là những góc
nhìn khác nhau. Không lấy khung nào trong 7 khung đầu/cuối, vì cửa sổ IMU 128 mẫu
đặt giữa khung đó sẽ tràn ra ngoài quỹ đạo và bước ghép sẽ bỏ nó. Các môi trường được xếp vào
`NUM_SHARDS` shard nặng xấp xỉ bằng nhau. **Mỗi lần chạy dựng một shard, mỗi shard là một Kaggle
dataset.**

Notebook không tải cả file zip. Nó đọc **mục lục** của zip rồi chỉ tải đúng những khung được chọn
(HTTP range), nên zip 35 GB của GreatMarsh chỉ tốn phần khung lấy từ nó.

**Baro.** TartanAir **không có** cảm biến áp suất. `baro.npy` là áp suất tĩnh **sạch** theo khí quyển
chuẩn ISA, tính từ độ cao thật: `pos_global` ở hệ NED nên *lên* = −z (đã kiểm: đạo hàm bậc hai của
`pos_global` khớp `acc_nograv` với tương quan 1,00, và trọng lực nằm trên +z). Độ cao nền và áp suất
mực nước biển được bốc ngẫu nhiên theo từng quỹ đạo (0–1.500 m, 101.325 ± 800 Pa), nên giá trị tuyệt
đối không để lộ đây là cảnh nào; chỉ phần thay đổi mới mang chuyển động dọc (~−12 Pa mỗi mét đi lên).
Nhiễu, trôi bias, trễ và tần số thấp hơn sẽ do stage corruption thêm vào, giống như với IMU.
`imu/baro.json` ghi lại tham số của từng quỹ đạo.

**Hai điều cần biết trước khi dùng.**
1. Code train hiện tại vẫn chạy ở **256×256**: `load_rgb` thu 640 → 256 (Lanczos) lúc đọc. Dataset
   mới dùng được ngay, nhưng muốn train ở độ phân giải cao hơn thì phải đổi config: `image_size`, và
   `focal_length_px` 128 → 320 nếu giữ nguyên 640. Train ảnh 640 nguyên thì tốn ~6,25× so với 256.
2. Đo trên 12 ảnh TartanAir: **tính trên mỗi pixel**, ảnh 640 gốc *mềm hơn* ảnh đã thu về 256 (độ dốc
   trung bình 0,062 so với 0,087). Cái 640 mang lại là **nhiều chi tiết hơn tổng thể** (vật nhỏ,
   texture xa), không phải từng pixel sắc hơn.

## Chạy
1. Kaggle → *New Notebook* → *File → Import Notebook* → chọn file này.
2. *Settings*: **Accelerator: None** (CPU, không tốn quota GPU), **Internet: On**.
3. **Chạy thử trước**: `PILOT = True` → *Run All*. Lượt này lấy 3 môi trường nhỏ, ~0,3 GB và hơn
   1.000 file. Nó đo tốc độ tải từ Kaggle, và cho biết Kaggle có lưu được output nhiều file hay không.
   Sau đó thử *Save Version* rồi *New Dataset* từ output.
4. **Chạy thật**: `PILOT = False`, `SHARD = 0` → *Save Version → Save & Run All (Commit)*. Khi xong,
   vào tab *Output* → **New Dataset** (ví dụ `tartanair640-s0`). Lặp lại với `SHARD = 1` và `SHARD = 2`.
   Dùng Commit, đừng lưu từ phiên tương tác: phiên tương tác có thể bị Kaggle khởi động lại, và
   `/kaggle/working` của phiên mới thì trống.
   **Mọi shard phải giữ nguyên** `NUM_SHARDS`, `SHARD_GB`, `FORMAT`, `SEED` và `SOURCE_REF`. Cell 2
   in ra một *fingerprint* của kế hoạch: fingerprint của ba shard phải giống nhau.
5. Nếu phiên tương tác bị ngắt giữa chừng, chạy lại Cell 3 trong cùng phiên: quỹ đạo đã xong được bỏ
   qua, khung đã ghi được giữ lại.

## Cell 1 — Lựa chọn, source, mạng

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request

REPO_URL = 'https://github.com/leesanghyoek/qwt_jepa_ver5.git'
SOURCE_REF = '0357db5'
PROJECT = Path('/tmp/qwt_jepa_version3')   # ngoài /kaggle/working: code không lọt vào dataset
OUT = Path('/kaggle/working/tartanair640')  # DATA_ROOT của dataset: con trực tiếp là các môi trường
META = Path('/kaggle/working/build_meta')   # kế hoạch, báo cáo, hình kiểm tra

PILOT = False       # True: 3 môi trường nhỏ, ~0,3 GB — thử tốc độ và cách Kaggle lưu output
SHARD = 0           # 0, 1, 2: mỗi lần Save Version dựng một shard
NUM_SHARDS = 3      # 3 × ~17 GB ≈ 52 GB. Nếu quota dataset riêng tư không đủ thì dùng 2
SHARD_GB = 18.0     # kế hoạch cho mỗi shard (trừ 5% dự phòng); Kaggle lưu tối đa 20 GB output
CAP_GB = 19.3       # chặn cứng: quỹ đạo nào làm shard vượt mức này thì bị bỏ, không làm hỏng lần lưu
FORMAT = 'webp'     # 'webp' lossless: pixel y hệt PNG, nhỏ hơn 24% | 'png': chép nguyên | 'jpeg98': q98, 45,9 dB
SOURCE = 'huggingface'  # 'airlab' là bản sao ở CMU (chậm hơn ~10× khi đo từ Việt Nam)
ENVIRONMENTS = None     # None = cả 74 môi trường
WORKERS = 12        # tiến trình tải + nén; CPU Kaggle có 4 lõi, số dư để che độ trễ mạng
PACK_ZIP = True     # Cell 5: mỗi môi trường một file zip (không nén thêm). Kaggle lưu ~25 file lớn nhanh
                    # hơn nhiều so với ~47 nghìn file nhỏ; lưu thư mục thì đặt False
SEED = 0

if PILOT:
    ENVIRONMENTS = ['AmericanDiner', 'RetroOffice', 'ArchVizTinyHouseNight']
    NUM_SHARDS, SHARD, SHARD_GB = 1, 0, 0.35

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--quiet', 'origin'], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--quiet', '--detach', SOURCE_REF], check=True)
for path in (str(PROJECT / 'tools'), str(PROJECT)):
    if path not in sys.path:
        sys.path.insert(0, path)
for name in [m for m in sys.modules if m == 'build_tartanair640' or m == 'qjepa' or m.startswith('qjepa.')]:
    del sys.modules[name]
import build_tartanair640 as builder
if ENVIRONMENTS is None:
    ENVIRONMENTS = list(builder.ENVIRONMENTS)

# Không bắt buộc. Token Hugging Face (Kaggle: Add-ons → Secrets, tên HF_TOKEN) nâng giới hạn request.
TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

source = builder.SOURCES[SOURCE]
try:
    builder.RemoteZip(source + 'AmericanDiner/Data_easy/imu.zip', TOKEN)
except Exception as error:
    raise SystemExit(f'Không đọc được {SOURCE} ({error}). Bật Settings → Internet: On rồi chạy lại.')
usage = shutil.disk_usage('/kaggle/working')
print('source', SOURCE_REF, '| môi trường', len(ENVIRONMENTS), '| shard', SHARD, '/', NUM_SHARDS,
      '| format', FORMAT, '| HF token', TOKEN is not None,
      f'| /kaggle/working trống {usage.free / 1e9:.0f} GB')

## Cell 2 — Kế hoạch: đọc mục lục mọi zip, đo tỉ lệ nén, chia khung và shard (chưa tải ảnh)

In [ ]:
started = time.perf_counter()
catalog, missing = builder.list_catalog(source, ENVIRONMENTS, token=TOKEN)
for problem in missing:
    print('thiếu:', problem)
available = sum(len(entry.frames) for entry in catalog.values())
print(f'{len(catalog)} quỹ đạo, {available:,} khung 640×640 có sẵn | {time.perf_counter() - started:.0f} s')

ratio = builder.calibrate(catalog, source, FORMAT, seed=SEED, token=TOKEN)
print(f'{FORMAT}: {ratio:.2f} byte cho mỗi byte PNG (đo trên 24 khung)')
plan = builder.make_plan(catalog, budget_bytes=NUM_SHARDS * SHARD_GB * 0.95e9, ratio=ratio, seed=SEED)
sizes = builder.environment_bytes(catalog, plan, ratio)
assignment = builder.assign_shards(sizes, NUM_SHARDS)
fingerprint = hashlib.sha256(json.dumps([plan, assignment, ratio], sort_keys=True).encode()).hexdigest()[:12]

rows = {}
for key, positions in plan.items():
    entry = catalog[key]
    row = rows.setdefault(entry.environment, [0, 0, 0])
    row[0] += 1
    row[1] += len(entry.frames)
    row[2] += len(positions)
print(f"\n{'môi trường':28s} {'quỹ đạo':>7s} {'có':>7s} {'giữ':>6s} {'cách (s)':>8s} {'GB':>6s} shard")
for environment in sorted(rows, key=lambda name: (assignment[name], name)):
    trajectories, frames, kept = rows[environment]
    print(f'{environment:28s} {trajectories:7d} {frames:7d} {kept:6d} {frames / kept / 10:8.1f} '
          f'{sizes[environment] / 1e9:6.2f} {assignment[environment]}')
loads = [sum(size for name, size in sizes.items() if assignment[name] == shard) for shard in range(NUM_SHARDS)]
kept_total = sum(len(positions) for positions in plan.values())
print(f'\ntổng: {kept_total:,} khung, {sum(loads) / 1e9:.1f} GB | mỗi shard (GB):',
      [round(load / 1e9, 2) for load in loads], '| fingerprint', fingerprint)
assert max(loads) <= CAP_GB * 1e9, 'Một shard ước tính vượt CAP_GB: giảm SHARD_GB hoặc tăng NUM_SHARDS.'

META.mkdir(parents=True, exist_ok=True)
(META / 'plan.json').write_text(json.dumps({
    'source_ref': SOURCE_REF, 'fingerprint': fingerprint, 'format': FORMAT, 'ratio': ratio,
    'num_shards': NUM_SHARDS, 'shard_gb': SHARD_GB, 'seed': SEED, 'margin': builder.DEFAULT_MARGIN,
    'environments': {name: {'shard': assignment[name], 'trajectories': row[0], 'frames_available': row[1],
                            'frames_kept': row[2], 'bytes': round(sizes[name])} for name, row in rows.items()},
    'missing': missing,
}, indent=2, ensure_ascii=False))

## Cell 3 — Dựng shard `SHARD`: tải đúng các khung đã chọn, nén, ghi IMU + baro

In [ ]:
report = builder.build_shard(
    catalog, plan, assignment, shard=SHARD, source=source, out_root=OUT, fmt=FORMAT, workers=WORKERS,
    cap_bytes=CAP_GB * 1e9, ratio=ratio, token=TOKEN, seed=SEED,
)
report['fingerprint'] = fingerprint
(META / f'shard{SHARD}.json').write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(f"\nshard {SHARD}: {report['trajectories']} quỹ đạo, {report['frames']:,} khung, "
      f"{report['stored_bytes'] / 1e9:.2f} GB, {report['seconds'] / 60:.0f} phút")
for item in report['skipped']:
    print('bỏ qua:', item['trajectory'], '—', item['skipped'])
print(f"/kaggle/working còn trống {shutil.disk_usage('/kaggle/working').free / 1e9:.1f} GB")

## Cell 4 — Kiểm tra bằng chính loader của repo

`build_manifest` là thứ bước train sẽ gọi: nó tìm quỹ đạo, ghép mỗi khung với cửa sổ IMU 128 mẫu
đặt giữa khung đó, và chia train/valid/test. Nếu mọi khung được ghép (`bị bỏ` = 0) thì dataset đúng
như loader cần.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from qjepa.data.manifest import build_manifest

manifest = build_manifest(OUT)
meta = manifest['meta']
rejected = {key: sum(item[key] for item in meta['pairing_audit'])
            for key in ('rejected_outside', 'rejected_centre', 'rejected_nonuniform')}
print('sample theo split:', meta['samples_per_split'], '| bị bỏ:', rejected)
print('quỹ đạo theo split:', {split: len(keys) for split, keys in meta['trajectories_per_split'].items()})

samples = [sample for split in manifest['samples'].values() for sample in split]
rng = np.random.default_rng(SHARD)
chosen = [samples[index] for index in rng.choice(len(samples), 8, replace=False)]
for sample in chosen:
    with Image.open(sample.image_path) as image:
        assert image.size == (640, 640), (sample.image_path, image.size)

figure = plt.figure(figsize=(16, 10))
for index, sample in enumerate(chosen):
    axis = figure.add_subplot(3, 4, index + 1)
    axis.imshow(Image.open(sample.image_path).convert('RGB'))
    axis.set_title(f'{sample.environment}\n{sample.difficulty} {sample.trajectory_id} khung {Path(sample.image_path).stem.split("_")[0]}', fontsize=8)
    axis.axis('off')
trajectory = Path(chosen[0].trajectory_path)
imu_time = np.load(trajectory / 'imu' / 'imu_time.npy')
seconds = imu_time - imu_time[0]
gyro = np.load(trajectory / 'imu' / 'gyro.npy')
pressure = np.load(trajectory / 'imu' / 'baro.npy')
height = -np.load(trajectory / 'imu' / 'pos_global.npy')[:, 2]
axis = figure.add_subplot(3, 2, 5)
axis.plot(seconds, gyro, linewidth=0.6)
axis.set_title(f'gyro (rad/s) — {chosen[0].environment}/{chosen[0].difficulty}/{chosen[0].trajectory_id}', fontsize=9)
axis.set_xlabel('s')
axis = figure.add_subplot(3, 2, 6)
axis.plot(seconds, pressure - pressure[0], color='tab:red', linewidth=0.8)
axis.set_ylabel('baro − baro[0] (Pa)', color='tab:red')
twin = axis.twinx()
twin.plot(seconds, height - height[0], color='tab:gray', linewidth=0.8)
twin.set_ylabel('độ cao thật − ban đầu (m)', color='tab:gray')
axis.set_title('baro tổng hợp so với độ cao thật: ngược chiều, ~12 Pa mỗi mét', fontsize=9)
axis.set_xlabel('s')
figure.tight_layout()
figure.savefig(META / f'preview_shard{SHARD}.png', dpi=80)
plt.show()

## Cell 5 — Đóng gói: mỗi môi trường một file zip

Chạy sau Cell 4, vì Cell 4 đọc thư mục. Mỗi môi trường được zip nguyên trạng (ảnh đã nén nên không
nén thêm), đọc lại toàn bộ để kiểm tra CRC, rồi mới xoá thư mục gốc. Output còn lại là
`tartanair640/<môi trường>.zip` (~25 file) và `build_meta/`. Khi tạo dataset, Kaggle giải nén mỗi zip
thành thư mục cùng tên, nên dataset đọc thành `tartanair640/<môi trường>/Data_easy/...`, không lặp tầng
nào.

In [ ]:
if PACK_ZIP:
    packed = builder.pack_environments(OUT)
    (META / f'pack_shard{SHARD}.json').write_text(json.dumps(packed, indent=2))
    print(f"{len(packed)} file zip, {sum(item['bytes'] for item in packed) / 1e9:.2f} GB")
print(sorted(os.listdir('/kaggle/working')), '|', sorted(os.listdir(OUT))[:5], '...')

## Sau khi dựng xong

**Tạo dataset.** *Save Version (Commit)* xong → tab *Output* → **New Dataset** (ví dụ
`tartanair640-s1`). Kaggle giải nén mỗi `<môi trường>.zip` thành thư mục cùng tên.

**Train trên cả ba shard.** Gắn cả ba dataset vào notebook train rồi gom chúng thành **một**
`DATA_ROOT`. `link_trajectories` tạo symlink `/tmp/tartanair640_all/<môi trường>/<độ khó>/<Pxxx>` tới
từng quỹ đạo, bất kể nó nằm sâu mấy tầng: shard 0 được đóng gói bằng tar nên có tầng lặp
`AbandonedCable/tartanair640/AbandonedCable/...`, và vẫn đọc được.

```python
from qjepa.data.tartanair import link_trajectories
roots = sorted(path for pattern in ('*/tartanair640*', '*/*/*/tartanair640*')
               for path in Path('/kaggle/input').glob(pattern) if path.is_dir())
DATA_ROOT = link_trajectories(roots, Path('/tmp/tartanair640_all'))
```

**Chia train/valid/test.** Đặt `data.split_rule: per_environment` trong config train. Mỗi môi trường
đưa 1 nhóm Pxxx vào test (nếu có ≥ 2 nhóm) và 1 nhóm vào valid (nếu có ≥ 5 nhóm); phần còn lại để
train. `Data_easy` và `Data_hard` của cùng một Pxxx luôn đi cùng nhau. ArchVizTinyHouse và OldBrickHouse
bản Day/Night bay **cùng một đường** với cùng tên Pxxx, nên hai bản đó cũng đi cùng nhau. Với cả ba
shard: train 423 nhóm (~73% khung), valid 64 nhóm (~12%, 64 môi trường), test 74 nhóm (~15%, đủ 74
môi trường). Split chỉ tính khi đã gắn **đủ ba shard**: tính trên một shard thì một số quỹ đạo sẽ
đổi split khi thêm shard sau.

**Dùng baro.** Loader hiện tại chỉ đọc `acc` và `gyro`, nên `baro.npy` nằm sẵn đó nhưng chưa được
dùng. Muốn model nhận baro thì phải thêm ba thứ: một kênh đầu vào cho nhánh IMU, thống kê chuẩn hoá
cho baro, và mô hình nhiễu baro trong `TrajectoryImuCorruptor`.